# US Revenue Forecast — 단계별 테스트 노트북

**소스 테이블** : `US_IS_from_FMP`  
**저장 테이블** : `us_revenue_forecast_data`  
**예측 모델**  : SARIMA · ETS · Prophet · LSTM · Theta  +  Ensemble (SARIMA+ETS+Theta 평균)

---
| 셀 번호 | 단계 |
|---------|------|
| Cell 1 | 환경 설정 & 경로 자동 감지 |
| Cell 2 | 모듈 Import |
| Cell 3 | 파라미터 설정 |
| Cell 4 | DB 연결 테스트 |
| Cell 5 | 재무 데이터 추출 함수 정의 |
| Cell 6 | 단일 티커 데이터 추출 테스트 |
| Cell 7 | 예측 함수 정의 |
| Cell 8 | 단일 티커 예측 테스트 |
| Cell 9 | Long-format 변환 함수 정의 |
| Cell 10 | Long-format 변환 테스트 |
| Cell 11 | DB 테이블 생성 & 저장 함수 정의 |
| Cell 12 | 단일 티커 저장 테스트 |
| Cell 13 | 배치 실행 (전체 / 특정 티커) |


## Cell 1 · 환경 설정 & 경로 자동 감지

노트북/데스크탑 어느 환경에서 실행해도 `DATA` 폴더를 자동으로 찾아 `sys.path`에 추가합니다.

In [1]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# ── 후보 루트 (노트북 / 데스크탑) ───────────────────────────
_CANDIDATE_ROOTS = [
    r"C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast",        # 노트북
    r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy",  # 데스크탑
]

def _setup_path() -> str:
    """
    프로젝트 루트(DATA 폴더의 부모)를 탐색해 sys.path에 추가합니다.
    탐색 순서:
      1) __file__ 기준 상위 폴더 중 DATA/ 를 포함하는 첫 번째 경로
      2) _CANDIDATE_ROOTS 에서 실존하는 첫 번째 경로
    """
    try:
        start = Path(__file__).resolve()
    except NameError:          # 노트북 환경 (__file__ 없음)
        start = Path.cwd()

    for p in [start] + list(start.parents):
        if (p / "DATA").is_dir():
            root = str(p)
            if root not in sys.path:
                sys.path.insert(0, root)
            print(f"[PATH] root (자동 감지): {root}")
            return root

    for candidate in _CANDIDATE_ROOTS:
        if os.path.isdir(candidate):
            if candidate not in sys.path:
                sys.path.insert(0, candidate)
            print(f"[PATH] root (후보 경로): {candidate}")
            return candidate

    raise EnvironmentError(
        "DATA 폴더를 찾을 수 없습니다. "
        "_CANDIDATE_ROOTS 목록을 현재 환경에 맞게 수정하세요."
    )

_root = _setup_path()
print(f"[확인] sys.path[0] = {sys.path[0]}")


[PATH] root (자동 감지): C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy
[확인] sys.path[0] = C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\US_Market\analysis\미국전기업_매출_예측


## Cell 2 · 모듈 Import

`DATA` 폴더 내 세 모듈을 불러옵니다.

In [2]:
# ── 내부 모듈 ───────────────────────────────────────────────
from DATA.config import get_db_info, get_engine, log
from DATA.us_target_ticker_list_2000 import ticker_list as DEFAULT_TICKER_LIST
from DATA.universal_ts_forecast_function_v2 import (
    forecast_sarima,
    forecast_ets,
    forecast_prophet,
    forecast_lstm,
    forecast_theta,
    infer_freq_alias,
    seasonal_periods_from_freq,
    clear_memory,
)

# ── 외부 라이브러리 ──────────────────────────────────────────
import numpy as np
import pandas as pd
from datetime import datetime
from sqlalchemy import text

print("[OK] 모든 모듈 Import 완료")
print(f"[OK] DEFAULT_TICKER_LIST 길이: {len(DEFAULT_TICKER_LIST)}개")


[OK] 모든 모듈 Import 완료
[OK] DEFAULT_TICKER_LIST 길이: 2000개


## Cell 3 · 파라미터 설정

여기서 항목·기간·모델 등을 자유롭게 변경하세요.

In [3]:
# ══════════════════════════════════════════
#  파라미터 — 필요에 따라 이 셀만 수정하세요
# ══════════════════════════════════════════

# 소스 / 저장 테이블
SRC_TABLE  = "US_IS_from_FMP"
DEST_TABLE = "us_revenue_forecast_data"

# 재무 항목  (sale / opi / ni / ebitda ...)
ITEM       = "sale"

# 예측 분기 수 (default 8)
HORIZON    = 8

# 최소 관측치 수 (default 28 = 7년 × 4분기)
MIN_OBS    = 28

# 사용할 예측 모델 목록
ALL_MODELS      = ["SARIMA", "ETS", "Prophet", "LSTM", "Theta"]
ENSEMBLE_MODELS = ["SARIMA", "ETS", "Theta"]   # 앙상블 구성 모델

# 배치 크기
BATCH_SIZE = 20

# 예측 실행일 (오늘)
FORECAST_DATE = datetime.now().strftime("%Y-%m-%d")

print("파라미터 설정 완료")
print(f"  ITEM={ITEM}, HORIZON={HORIZON}, MIN_OBS={MIN_OBS}")
print(f"  MODELS={ALL_MODELS}")
print(f"  FORECAST_DATE={FORECAST_DATE}")


파라미터 설정 완료
  ITEM=sale, HORIZON=8, MIN_OBS=28
  MODELS=['SARIMA', 'ETS', 'Prophet', 'LSTM', 'Theta']
  FORECAST_DATE=2026-03-24


## Cell 4 · DB 연결 테스트

In [4]:
db_info = get_db_info()
engine  = get_engine(db_info)

try:
    with engine.connect() as conn:
        result = conn.execute(text("SELECT 1"))
    print("[OK] DB 연결 성공")
    print(f"     host={db_info['host']}  port={db_info['port']}  db={db_info['database']}")
except Exception as e:
    print(f"[FAIL] DB 연결 실패: {e}")


[OK] DB 연결 성공
     host=192.168.0.230  port=3307  db=investar


## Cell 5 · 재무 데이터 추출 함수 정의

`US_IS_from_FMP` → ticker + item 기준 분기 시계열 추출  

**처리 흐름**
1. 날짜 파싱 및 오름차순 정렬  
2. 월별 중복 제거 (같은 월 → 마지막 행 사용, 단일 행은 보존)  
3. 분기 단위 재집계 (같은 분기 → 마지막 행, 분기말 날짜로 통일)  
4. MIN_OBS 미달 시 ValueError 발생


In [5]:
def fetch_financial_series(
    engine,
    ticker: str,
    item: str = "sale",
    min_obs: int = 28,
) -> pd.DataFrame:
    """
    US_IS_from_FMP 테이블에서 ticker + item 기준으로 분기 시계열을 추출합니다.

    반환 컬럼: date, report_date, period, date_month, value
    """
    query = text("""
        SELECT
            date,
            report_date,
            period,
            date_month,
            value
        FROM   US_IS_from_FMP
        WHERE  ticker = :ticker
          AND  item   = :item
          AND  value  IS NOT NULL
        ORDER  BY date
    """)

    with engine.connect() as conn:
        df = pd.read_sql(query, conn, params={"ticker": ticker, "item": item})

    if df.empty:
        raise ValueError(f"[{ticker}] '{item}' 데이터 없음")

    # 1. 날짜 변환
    df["date"]        = pd.to_datetime(df["date"],        errors="coerce")
    df["report_date"] = pd.to_datetime(df["report_date"], errors="coerce")
    df["date_month"]  = pd.to_datetime(df["date_month"],  errors="coerce")
    df = df.dropna(subset=["date"]).sort_values("date").reset_index(drop=True)
    df["value"] = df["value"].astype(float)

    # 2. 월별 중복 제거
    #    같은 월에 여러 행 → 마지막 행 사용
    #    처음/끝처럼 해당 월에 1건만 있는 행은 그대로 보존
    df["_ym"]  = df["date"].dt.to_period("M")
    dup_mask   = df["_ym"].duplicated(keep=False)

    if dup_mask.any():
        df_single = df[~dup_mask]
        df_dup    = (
            df[dup_mask]
            .groupby("_ym", sort=True)
            .last()
            .reset_index()
        )
        df = (
            pd.concat([df_single, df_dup], ignore_index=True)
            .sort_values("date")
            .reset_index(drop=True)
        )
    df = df.drop(columns=["_ym"])

    # 3. 분기 단위 재집계 → 분기말 날짜 통일
    df["_q"] = df["date"].dt.to_period("Q")
    df = (
        df.groupby("_q", sort=True)
          .last()
          .reset_index()
    )
    df["date"] = df["_q"].dt.to_timestamp("Q")   # 예: 2023Q4 → 2023-12-31
    df = (
        df.drop(columns=["_q"])
          .sort_values("date")
          .reset_index(drop=True)
    )

    # 4. 최소 관측치 검사
    if len(df) < min_obs:
        raise ValueError(
            f"[{ticker}] '{item}' 관측치 부족: {len(df)}개 < 최소 {min_obs}개"
        )

    return df   # columns: date, report_date, period, date_month, value

print("[OK] fetch_financial_series 함수 정의 완료")


[OK] fetch_financial_series 함수 정의 완료


## Cell 6 · 단일 티커 데이터 추출 테스트

`TEST_TICKER`를 원하는 티커로 변경해서 테스트하세요.

In [6]:
TEST_TICKER = "AAPL"   # ← 테스트할 티커 변경

try:
    src_df = fetch_financial_series(engine, TEST_TICKER, item=ITEM, min_obs=MIN_OBS)
    print(f"[OK] {TEST_TICKER} '{ITEM}' 데이터 추출 성공: {len(src_df)}분기")
    print(f"     기간: {src_df['date'].iloc[0].date()} ~ {src_df['date'].iloc[-1].date()}")
    display(src_df.tail(8))
except Exception as e:
    print(f"[FAIL] {e}")


[OK] AAPL 'sale' 데이터 추출 성공: 45분기
     기간: 2015-03-31 ~ 2026-03-31


,date,report_date,period,date_month,value
37,2024-06-30,2024-06-29,Q3,2024-06-01,8.577700e+10
38,2024-09-30,2024-09-28,Q4,2024-09-01,9.493000e+10
39,2024-12-31,2024-12-28,Q1,2024-12-01,1.243000e+11
40,2025-03-31,2025-03-29,Q2,2025-03-01,9.535900e+10
41,2025-06-30,2025-06-28,Q3,2025-06-01,9.403600e+10
42,2025-09-30,2025-09-27,Q4,2025-09-01,1.024660e+11
43,2025-12-31,2025-12-27,Q1,2025-12-01,1.437560e+11
44,2026-03-31,2025-12-27,Q1,2025-12-01,1.437560e+11


## Cell 7 · 예측 함수 정의

단일 시계열에 대해 지정 모델 목록으로 예측을 수행합니다.

In [8]:
def make_forecast_index(
    last_date: pd.Timestamp,
    horizon: int,
    freq: str,
) -> pd.DatetimeIndex:
    """last_date 이후 horizon 개의 분기말(또는 월말) 날짜 생성"""
    offset = pd.offsets.QuarterEnd(1) if freq == "Q" else pd.offsets.MonthEnd(1)
    dates, cur = [], last_date
    for _ in range(horizon):
        cur = cur + offset
        dates.append(cur)
    return pd.DatetimeIndex(dates)


def forecast_one_ticker(
    y: pd.Series,
    ticker: str,
    horizon: int,
    models: list,
) -> dict:
    """
    단일 시계열에 대해 지정 모델로 예측 수행.

    Parameters
    ----------
    y       : pd.Series (index=분기말 DatetimeIndex, values=재무값)
    ticker  : 종목 코드 (로그 출력용)
    horizon : 예측 분기 수
    models  : 사용할 모델 리스트

    Returns
    -------
    dict : { 'SARIMA': {...}, 'ETS': {...}, ... }
    """
    freq = infer_freq_alias(y.index)
    m    = seasonal_periods_from_freq(freq)
    results = {}

    for model_name in models:
        log(ticker, f"  [{model_name}] 시작")
        try:
            if model_name == "SARIMA":
                res = forecast_sarima(y, horizon, seasonal_period=m)
            elif model_name == "ETS":
                res = forecast_ets(y, horizon, m=m)
            elif model_name == "Prophet":
                res = forecast_prophet(y, horizon, m=m)
            elif model_name == "LSTM":
                res = forecast_lstm(y, horizon)
            elif model_name == "Theta":
                res = forecast_theta(y, horizon, m=m)
            else:
                res = {"error": f"알 수 없는 모델: {model_name}"}

            status = "완료" if "error" not in res else f"실패({res['error']})"
            log(ticker, f"  [{model_name}] {status}")
            results[model_name] = res

        except Exception as e:
            log(ticker, f"  [{model_name}] 예외: {e}")
            results[model_name] = {"error": str(e)}

        clear_memory()

    return results

print("[OK] forecast_one_ticker / make_forecast_index 함수 정의 완료")


[OK] forecast_one_ticker / make_forecast_index 함수 정의 완료


## Cell 8 · 단일 티커 예측 테스트

Cell 6에서 추출한 `src_df`로 예측을 실행합니다.

In [9]:
# Cell 6 의 src_df 사용 (TEST_TICKER)
y = src_df.set_index("date")["value"]
y.index = pd.DatetimeIndex(y.index)
y.name  = ITEM

print(f"예측 입력 시계열: {len(y)}분기  ({y.index[0].date()} ~ {y.index[-1].date()})")

# 테스트용으로 모델 1개만 먼저 실행 → 확인 후 ALL_MODELS 로 변경 가능
TEST_MODELS = ["SARIMA", "ETS", "Theta"]   # ← 원하는 모델로 변경

forecast_results = forecast_one_ticker(y, TEST_TICKER, HORIZON, TEST_MODELS)

# 결과 요약 출력
freq           = infer_freq_alias(y.index)
forecast_index = make_forecast_index(y.index[-1], HORIZON, freq)

print("\n[예측 결과 요약]")
for model_name, res in forecast_results.items():
    if "error" in res:
        print(f"  {model_name:<10}: FAIL — {res['error']}")
    else:
        fc  = np.asarray(res["forecast"])
        msg = f"  {model_name:<10}: {fc.round(2).tolist()}"
        if model_name == "SARIMA":
            spec = res.get("spec", {})
            msg += f"  | order={spec.get('order')} seasonal={spec.get('seasonal_order')} AIC={spec.get('ic_value', ''):.2f}" if spec.get('ic_value') else ""
        print(msg)


예측 입력 시계열: 45분기  (2015-03-31 ~ 2026-03-31)
[AAPL]   [SARIMA] 시작
[메모리] forecast_sarima 실행 전: 432.34 MB
[메모리] find_best_sarima_params 실행 전: 432.36 MB
[메모리] find_best_sarima_params 실행 후: 436.80 MB (변화: +4.44 MB)
[메모리] forecast_sarima 실행 후: 436.85 MB (변화: +4.51 MB)
[AAPL]   [SARIMA] 완료
[AAPL]   [ETS] 시작
[메모리] forecast_ets 실행 전: 436.92 MB
[메모리] forecast_ets 실행 후: 437.12 MB (변화: +0.20 MB)
[AAPL]   [ETS] 완료
[AAPL]   [Theta] 시작
[메모리] forecast_theta 실행 전: 437.12 MB
[메모리] forecast_theta 실행 후: 437.35 MB (변화: +0.23 MB)
[AAPL]   [Theta] 완료

[예측 결과 요약]
  SARIMA    : [135238172626.34, 145630469495.55, 190965969155.86, 168115103283.58, 159595361653.3, 171899902805.69, 222578777790.81, 198600825393.47]  | order=(1, 0, 0) seasonal=(1, 0, 1, 4) AIC=-49.16
  ETS       : [111600659044.97, 119882190949.01, 175206371708.67, 138591237536.66, 120227487134.83, 129149188663.92, 188749976754.62, 149304460838.56]
  Theta     : [115136490466.84, 122886617885.43, 181007559375.48, 137558352814.78, 119548528459.55, 12

## Cell 9 · Long-format 변환 함수 정의

actual + forecast(각 모델) + Ensemble → 하나의 long-format DataFrame

**저장 컬럼**

| 컬럼 | 설명 |
|------|------|
| ticker | 종목 코드 |
| item | 재무 항목 |
| date | 기준일(분기말) |
| period | 회계 분기(Q1~Q4/FY) — actual 행만 |
| date_month | 해당 분기 기간 — actual 행만 |
| data_type | `actual` / `forecast` |
| model | actual / SARIMA / ETS / Prophet / LSTM / Theta / Ensemble |
| value | 수치값 |
| forecast_date | 예측 실행일 |
| sarima_order | SARIMA (p,d,q) — SARIMA 행만 |
| sarima_seasonal_order | SARIMA (P,D,Q,m) — SARIMA 행만 |
| sarima_ic_value | SARIMA 최적 AIC — SARIMA 행만 |
| created_at | 레코드 생성 시각 |


In [11]:
def build_long_df(
    ticker: str,
    item: str,
    src_df: pd.DataFrame,
    forecast_results: dict,
    forecast_index: pd.DatetimeIndex,
    forecast_date: str,
    ensemble_models: list = None,
) -> pd.DataFrame:
    """
    실제값(actual) + 예측값(forecast) + 앙상블 → long-format DataFrame.
    """
    if ensemble_models is None:
        ensemble_models = ENSEMBLE_MODELS

    now_str = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    rows = []

    # ── actual 행 ────────────────────────────────────────────
    for _, row in src_df.iterrows():
        rows.append({
            "ticker"                : ticker,
            "item"                  : item,
            "date"                  : row["date"].strftime("%Y-%m-%d"),
            "period"                : row.get("period"),
            "date_month"            : (
                row["date_month"].strftime("%Y-%m-%d")
                if pd.notna(row.get("date_month")) else None
            ),
            "data_type"             : "actual",
            "model"                 : "actual",
            "value"                 : round(float(row["value"]), 6),
            "forecast_date"         : forecast_date,
            "sarima_order"          : None,
            "sarima_seasonal_order" : None,
            "sarima_ic_value"       : None,
            "created_at"            : now_str,
        })

    # ── forecast 행 ──────────────────────────────────────────
    ensemble_bucket = {}   # { date_str : [val, ...] }

    for model_name, res in forecast_results.items():
        if "error" in res or "forecast" not in res:
            continue

        fc_arr = np.asarray(res["forecast"])
        spec   = res.get("spec", {})

        sarima_order    = None
        sarima_seasonal = None
        sarima_ic       = None
        if model_name == "SARIMA":
            sarima_order    = str(spec.get("order", ""))
            sarima_seasonal = str(spec.get("seasonal_order", ""))
            raw_ic = spec.get("ic_value")
            if raw_ic is not None and np.isfinite(float(raw_ic)):
                sarima_ic = round(float(raw_ic), 4)

        for i, dt in enumerate(forecast_index):
            if i >= len(fc_arr):
                break
            val    = float(fc_arr[i])
            dt_str = dt.strftime("%Y-%m-%d")

            rows.append({
                "ticker"                : ticker,
                "item"                  : item,
                "date"                  : dt_str,
                "period"                : None,
                "date_month"            : None,
                "data_type"             : "forecast",
                "model"                 : model_name,
                "value"                 : round(val, 6),
                "forecast_date"         : forecast_date,
                "sarima_order"          : sarima_order,
                "sarima_seasonal_order" : sarima_seasonal,
                "sarima_ic_value"       : sarima_ic,
                "created_at"            : now_str,
            })

            if model_name in ensemble_models:
                ensemble_bucket.setdefault(dt_str, []).append(val)

    # ── Ensemble 행 (SARIMA + ETS + Theta 평균) ───────────────
    for dt_str, vals in ensemble_bucket.items():
        rows.append({
            "ticker"                : ticker,
            "item"                  : item,
            "date"                  : dt_str,
            "period"                : None,
            "date_month"            : None,
            "data_type"             : "forecast",
            "model"                 : "Ensemble",
            "value"                 : round(float(np.mean(vals)), 6),
            "forecast_date"         : forecast_date,
            "sarima_order"          : None,
            "sarima_seasonal_order" : None,
            "sarima_ic_value"       : None,
            "created_at"            : now_str,
        })

    return pd.DataFrame(rows)

print("[OK] build_long_df 함수 정의 완료")


[OK] build_long_df 함수 정의 완료


## Cell 10 · Long-format 변환 테스트

In [12]:
long_df = build_long_df(
    ticker           = TEST_TICKER,
    item             = ITEM,
    src_df           = src_df,
    forecast_results = forecast_results,
    forecast_index   = forecast_index,
    forecast_date    = FORECAST_DATE,
)

print(f"[OK] long_df 생성: {len(long_df)}행")
print("\n모델별 행 수:")
display(long_df.groupby(["data_type", "model"]).size().reset_index(name="rows"))
print("\n샘플 (forecast 상위 5행):")
display(long_df[long_df["data_type"] == "forecast"].head())


[OK] long_df 생성: 77행

모델별 행 수:


,data_type,model,rows
0,actual,actual,45
1,forecast,ETS,8
2,forecast,Ensemble,8
3,forecast,SARIMA,8
4,forecast,Theta,8



샘플 (forecast 상위 5행):


,ticker,item,date,period,date_month,data_type,model,value,forecast_date,sarima_order,sarima_seasonal_order,sarima_ic_value,created_at
45,AAPL,sale,2026-06-30,None,None,forecast,SARIMA,1.352382e+11,2026-03-24,"(1, 0, 0)","(1, 0, 1, 4)",-49.1557,2026-03-24 22:50:46
46,AAPL,sale,2026-09-30,None,None,forecast,SARIMA,1.456305e+11,2026-03-24,"(1, 0, 0)","(1, 0, 1, 4)",-49.1557,2026-03-24 22:50:46
47,AAPL,sale,2026-12-31,None,None,forecast,SARIMA,1.909660e+11,2026-03-24,"(1, 0, 0)","(1, 0, 1, 4)",-49.1557,2026-03-24 22:50:46
48,AAPL,sale,2027-03-31,None,None,forecast,SARIMA,1.681151e+11,2026-03-24,"(1, 0, 0)","(1, 0, 1, 4)",-49.1557,2026-03-24 22:50:46
49,AAPL,sale,2027-06-30,None,None,forecast,SARIMA,1.595954e+11,2026-03-24,"(1, 0, 0)","(1, 0, 1, 4)",-49.1557,2026-03-24 22:50:46


## Cell 11 · DB 테이블 생성 & 저장 함수 정의

In [17]:
def ensure_table(engine) -> None:
    """us_revenue_forecast_data 테이블이 없으면 생성합니다."""
    ddl = f"""
    CREATE TABLE IF NOT EXISTS {DEST_TABLE} (
        id                    BIGINT        NOT NULL AUTO_INCREMENT,
        ticker                VARCHAR(20)   NOT NULL,
        item                  VARCHAR(50)   NOT NULL  COMMENT '재무항목(sale/opi/ni...)',
        date                  DATE          NOT NULL  COMMENT '기준일(분기말)',
        period                VARCHAR(10)            COMMENT '회계분기(Q1~Q4/FY, actual만)',
        date_month            DATE                   COMMENT '해당분기 기간(actual만)',
        data_type             VARCHAR(10)   NOT NULL  COMMENT 'actual | forecast',
        model                 VARCHAR(20)   NOT NULL  COMMENT 'actual/SARIMA/ETS/Prophet/LSTM/Theta/Ensemble',
        value                 DOUBLE,
        forecast_date         DATE          NOT NULL  COMMENT '예측 실행일',
        sarima_order          VARCHAR(30)            COMMENT 'SARIMA (p,d,q)',
        sarima_seasonal_order VARCHAR(30)            COMMENT 'SARIMA (P,D,Q,m)',
        sarima_ic_value       DOUBLE                 COMMENT 'SARIMA 최적 AIC',
        created_at            DATETIME      NOT NULL,
        PRIMARY KEY (id),
        UNIQUE KEY uq_row (ticker, item, date, model, forecast_date),
        INDEX idx_ticker_date   (ticker, date),
        INDEX idx_forecast_date (forecast_date)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
      COMMENT='US 재무항목 시계열 예측 결과 (long-format)';
    """
    with engine.connect() as conn:
        conn.execute(text(ddl))
        conn.commit()
    log("DB", f"{DEST_TABLE} 테이블 확인/생성 완료")


def _clean_val(v):
    """float NaN / inf / pandas NaT → None (MySQL NULL 변환)"""
    if v is None:
        return None
    # pandas NaT 처리 (date 변환 후 남는 NaT)
    try:
        if pd.isna(v):
            return None
    except (TypeError, ValueError):
        pass
    # float NaN / inf 처리
    try:
        if isinstance(v, float) and (v != v or v == float("inf") or v == float("-inf")):
            return None
    except Exception:
        pass
    return v


def save_to_db(engine, df: pd.DataFrame) -> int:
    """
    UNIQUE KEY (ticker, item, date, model, forecast_date) 기준으로
    중복을 제외하고 신규 행만 INSERT합니다.
    float NaN / inf 값은 MySQL NULL로 변환합니다.
    """
    if df.empty:
        return 0

    tickers = df["ticker"].unique().tolist()
    items   = df["item"].unique().tolist()
    fdates  = df["forecast_date"].unique().tolist()

    t_ph = ",".join([f"'{v}'" for v in tickers])
    i_ph = ",".join([f"'{v}'" for v in items])
    d_ph = ",".join([f"'{v}'" for v in fdates])

    chk_sql = text(f"""
        SELECT ticker, item, date, model, forecast_date
        FROM   {DEST_TABLE}
        WHERE  ticker        IN ({t_ph})
          AND  item          IN ({i_ph})
          AND  forecast_date IN ({d_ph})
    """)

    with engine.connect() as conn:
        existing = pd.read_sql(chk_sql, conn)

    if not existing.empty:
        existing["date"]          = pd.to_datetime(existing["date"]).dt.strftime("%Y-%m-%d")
        existing["forecast_date"] = pd.to_datetime(existing["forecast_date"]).dt.strftime("%Y-%m-%d")
        key_cols  = ["ticker", "item", "date", "model", "forecast_date"]
        exist_set = set(zip(*[existing[c].tolist() for c in key_cols]))
        df_new = df[
            ~df.apply(
                lambda r: (
                    r["ticker"], r["item"], r["date"],
                    r["model"], r["forecast_date"]
                ) in exist_set,
                axis=1,
            )
        ]
    else:
        df_new = df.copy()

    if df_new.empty:
        log("DB", "신규 데이터 없음 (전부 중복 스킵)")
        return 0

    # ── 날짜 타입 정리 ────────────────────────────────────────
    df_new = df_new.copy()
    for col in ["date", "forecast_date", "date_month"]:
        df_new[col] = pd.to_datetime(df_new[col], errors="coerce").dt.date

    # ── float NaN / inf → None (PyMySQL은 nan을 직접 거부) ────
    cols = [
        "ticker", "item", "date", "period", "date_month",
        "data_type", "model", "value", "forecast_date",
        "sarima_order", "sarima_seasonal_order", "sarima_ic_value",
        "created_at",
    ]
    records = [
        {k: _clean_val(v) for k, v in row.items()}
        for row in df_new[cols].to_dict(orient="records")
    ]

    insert_sql = text(f"""
        INSERT INTO {DEST_TABLE}
            (ticker, item, date, period, date_month,
             data_type, model, value, forecast_date,
             sarima_order, sarima_seasonal_order, sarima_ic_value,
             created_at)
        VALUES
            (:ticker, :item, :date, :period, :date_month,
             :data_type, :model, :value, :forecast_date,
             :sarima_order, :sarima_seasonal_order, :sarima_ic_value,
             :created_at)
    """)

    with engine.connect() as conn:
        conn.execute(insert_sql, records)
        conn.commit()

    log("DB", f"{len(records)}행 INSERT 완료")
    return len(records)

print("[OK] ensure_table / save_to_db 함수 재정의 완료")

[OK] ensure_table / save_to_db 함수 재정의 완료


## Cell 12 · 단일 티커 저장 테스트

Cell 10의 `long_df`를 DB에 저장합니다.

In [18]:
# 테이블 생성 (없으면)
ensure_table(engine)

# 저장
inserted = save_to_db(engine, long_df)
print(f"[OK] {TEST_TICKER} 저장 완료: {inserted}행 INSERT")

# 저장 확인
with engine.connect() as conn:
    check = pd.read_sql(
        text(f"""
            SELECT model, data_type, COUNT(*) as cnt
            FROM   {DEST_TABLE}
            WHERE  ticker = '{TEST_TICKER}'
              AND  item   = '{ITEM}'
              AND  forecast_date = '{FORECAST_DATE}'
            GROUP  BY model, data_type
            ORDER  BY data_type, model
        """),
        conn
    )
print("\n[DB 저장 확인]")
display(check)


[DB] us_revenue_forecast_data 테이블 확인/생성 완료
[DB] 77행 INSERT 완료
[OK] AAPL 저장 완료: 77행 INSERT

[DB 저장 확인]


,model,data_type,cnt
0,actual,actual,45
1,Ensemble,forecast,8
2,ETS,forecast,8
3,SARIMA,forecast,8
4,Theta,forecast,8


## Cell 13 · 배치 실행

- `RUN_TICKERS = None` → 전체 `DEFAULT_TICKER_LIST` 예측  
- `RUN_TICKERS = ["AAPL", "MSFT", ...]` → 특정 티커만 예측  
- `RUN_MODELS` 에서 사용할 모델 선택 가능


In [19]:
# ── 배치 설정 ──────────────────────────────────────────────
RUN_TICKERS = ["AAPL", "MSFT", "NVDA"]   # None 이면 전체 실행
RUN_MODELS  = ALL_MODELS                  # 또는 ["SARIMA", "ETS", "Theta"]
RUN_ITEM    = ITEM                        # Cell 3 에서 설정한 값 사용
RUN_HORIZON = HORIZON
RUN_MIN_OBS = MIN_OBS

# ── 배치 실행 ───────────────────────────────────────────────
tickers = RUN_TICKERS if RUN_TICKERS else DEFAULT_TICKER_LIST
total   = len(tickers)

ensure_table(engine)

success, skipped, errored = 0, 0, 0
skip_list, error_list     = [], []

log("BATCH", "=" * 64)
log("BATCH", f"시작  | 티커 {total}개 | 항목: {RUN_ITEM} | 예측기간: {RUN_HORIZON}분기")
log("BATCH", f"모델  : {RUN_MODELS}")
log("BATCH", f"예측일: {FORECAST_DATE} | min_obs: {RUN_MIN_OBS}")
log("BATCH", "=" * 64)

for i, ticker in enumerate(tickers, 1):
    pct = i / total * 100
    log("PROGRESS", f"[{i:>4}/{total}] ({pct:5.1f}%)  >>  {ticker}")

    # 1. 데이터 추출
    try:
        src_df = fetch_financial_series(engine, ticker, RUN_ITEM, RUN_MIN_OBS)
    except Exception as e:
        log(ticker, f"[SKIP] {e}")
        skipped += 1
        skip_list.append(ticker)
        continue

    y = src_df.set_index("date")["value"]
    y.index = pd.DatetimeIndex(y.index)
    y.name  = RUN_ITEM
    log(ticker, f"  {len(y)}분기 | {y.index[0].date()} ~ {y.index[-1].date()}")

    # 2. 예측
    try:
        fc_results = forecast_one_ticker(y, ticker, RUN_HORIZON, RUN_MODELS)
    except Exception as e:
        log(ticker, f"[ERROR] 예측: {e}")
        errored += 1; error_list.append(ticker); continue

    # 3. 예측 인덱스
    freq = infer_freq_alias(y.index)
    fc_index = make_forecast_index(y.index[-1], RUN_HORIZON, freq)

    # 4. Long-format 변환
    try:
        ldf = build_long_df(
            ticker           = ticker,
            item             = RUN_ITEM,
            src_df           = src_df,
            forecast_results = fc_results,
            forecast_index   = fc_index,
            forecast_date    = FORECAST_DATE,
        )
    except Exception as e:
        log(ticker, f"[ERROR] 변환: {e}")
        errored += 1; error_list.append(ticker); continue

    # 5. DB 저장
    try:
        save_to_db(engine, ldf)
        success += 1
    except Exception as e:
        log(ticker, f"[ERROR] 저장: {e}")
        errored += 1; error_list.append(ticker); continue

    clear_memory()

# 요약
log("BATCH", "=" * 64)
log("BATCH", f"완료 | 성공: {success}  스킵: {skipped}  오류: {errored}")
if skip_list:  log("BATCH", f"스킵 티커 : {skip_list}")
if error_list: log("BATCH", f"오류 티커 : {error_list}")
log("BATCH", "=" * 64)


[DB] us_revenue_forecast_data 테이블 확인/생성 완료
[BATCH] ================================================================
[BATCH] 시작  | 티커 3개 | 항목: sale | 예측기간: 8분기
[BATCH] 모델  : ['SARIMA', 'ETS', 'Prophet', 'LSTM', 'Theta']
[BATCH] 예측일: 2026-03-24 | min_obs: 28
[BATCH] ================================================================
[PROGRESS] [   1/3] ( 33.3%)  >>  AAPL
[AAPL]   45분기 | 2015-03-31 ~ 2026-03-31
[AAPL]   [SARIMA] 시작
[메모리] forecast_sarima 실행 전: 464.79 MB
[메모리] find_best_sarima_params 실행 전: 464.79 MB
[메모리] find_best_sarima_params 실행 후: 464.98 MB (변화: +0.19 MB)
[메모리] forecast_sarima 실행 후: 464.98 MB (변화: +0.19 MB)
[AAPL]   [SARIMA] 완료
[AAPL]   [ETS] 시작
[메모리] forecast_ets 실행 전: 464.98 MB
[메모리] forecast_ets 실행 후: 464.98 MB (변화: +0.00 MB)
[AAPL]   [ETS] 완료
[AAPL]   [Prophet] 시작
[메모리] forecast_prophet 실행 전: 464.98 MB


22:56:22 - cmdstanpy - INFO - Chain [1] start processing
22:56:23 - cmdstanpy - INFO - Chain [1] done processing


[메모리] forecast_prophet 실행 후: 465.59 MB (변화: +0.61 MB)
[AAPL]   [Prophet] 완료
[AAPL]   [LSTM] 시작
[메모리] forecast_lstm 실행 전: 465.59 MB
[메모리] forecast_lstm 실행 후: 1462.48 MB (변화: +996.89 MB)
[경고] 메모리 사용량이 크게 증가했습니다. 메모리 정리를 권장합니다.
[AAPL]   [LSTM] 완료
[AAPL]   [Theta] 시작
[메모리] forecast_theta 실행 전: 1462.48 MB
[메모리] forecast_theta 실행 후: 1462.48 MB (변화: +0.00 MB)
[AAPL]   [Theta] 완료
[DB] 16행 INSERT 완료
[PROGRESS] [   2/3] ( 66.7%)  >>  MSFT
[MSFT]   45분기 | 2015-03-31 ~ 2026-03-31
[MSFT]   [SARIMA] 시작
[메모리] forecast_sarima 실행 전: 1462.50 MB
[메모리] find_best_sarima_params 실행 전: 1462.50 MB
[메모리] find_best_sarima_params 실행 후: 1465.20 MB (변화: +2.70 MB)
[메모리] forecast_sarima 실행 후: 1465.20 MB (변화: +2.70 MB)
[MSFT]   [SARIMA] 완료
[MSFT]   [ETS] 시작
[메모리] forecast_ets 실행 전: 1465.20 MB
[메모리] forecast_ets 실행 후: 1465.45 MB (변화: +0.25 MB)
[MSFT]   [ETS] 완료
[MSFT]   [Prophet] 시작


22:56:47 - cmdstanpy - INFO - Chain [1] start processing


[메모리] forecast_prophet 실행 전: 1465.45 MB


...

[메모리] forecast_prophet 실행 후: 1466.98 MB (변화: +1.53 MB)
[MSFT]   [Prophet] 완료
[MSFT]   [LSTM] 시작
[메모리] forecast_lstm 실행 전: 1466.98 MB
[메모리] forecast_lstm 실행 후: 1473.19 MB (변화: +6.21 MB)
[MSFT]   [LSTM] 완료
[MSFT]   [Theta] 시작
[메모리] forecast_theta 실행 전: 1473.19 MB
[메모리] forecast_theta 실행 후: 1473.20 MB (변화: +0.01 MB)
[MSFT]   [Theta] 완료
[DB] 93행 INSERT 완료
[PROGRESS] [   3/3] (100.0%)  >>  NVDA
[NVDA]   45분기 | 2015-03-31 ~ 2026-03-31
[NVDA]   [SARIMA] 시작
[메모리] forecast_sarima 실행 전: 1473.28 MB
[메모리] find_best_sarima_params 실행 전: 1473.28 MB
[메모리] find_best_sarima_params 실행 후: 1475.09 MB (변화: +1.81 MB)
[메모리] forecast_sarima 실행 후: 1475.09 MB (변화: +1.81 MB)
[NVDA]   [SARIMA] 완료
[NVDA]   [ETS] 시작
[메모리] forecast_ets 실행 전: 1475.09 MB
[메모리] forecast_ets 실행 후: 1475.32 MB (변화: +0.23 MB)
[NVDA]   [ETS] 완료
[NVDA]   [Prophet] 시작


22:57:08 - cmdstanpy - INFO - Chain [1] start processing


[메모리] forecast_prophet 실행 전: 1475.32 MB


22:57:08 - cmdstanpy - INFO - Chain [1] done processing


[메모리] forecast_prophet 실행 후: 1477.30 MB (변화: +1.98 MB)
[NVDA]   [Prophet] 완료
[NVDA]   [LSTM] 시작
[메모리] forecast_lstm 실행 전: 1477.30 MB
[메모리] forecast_lstm 실행 후: 1475.69 MB (변화: -1.61 MB)
[NVDA]   [LSTM] 완료
[NVDA]   [Theta] 시작
[메모리] forecast_theta 실행 전: 1475.69 MB
[메모리] forecast_theta 실행 후: 1475.70 MB (변화: +0.00 MB)
[NVDA]   [Theta] 완료
[DB] 93행 INSERT 완료
[BATCH] ================================================================
[BATCH] 완료 | 성공: 3  스킵: 0  오류: 0
[BATCH] ================================================================


## Cell 14 · 저장 결과 조회 (선택)

In [21]:
# 배치 결과 요약 조회
with engine.connect() as conn:
    summary = pd.read_sql(
        text(f"""
            SELECT
                ticker,
                item,
                model,
                MIN(date)  AS date_from,
                MAX(date)  AS date_to,
                COUNT(*)   AS row_count,
                forecast_date
            FROM   {DEST_TABLE}
            WHERE  forecast_date = '{FORECAST_DATE}'
            GROUP  BY ticker, item, model, forecast_date
            ORDER  BY ticker, model
        """),
        conn
    )
print(f"저장 결과: {len(summary)}건")
display(summary)


저장 결과: 21건


,ticker,item,model,date_from,date_to,row_count,forecast_date
0,AAPL,sale,actual,2015-03-31,2026-03-31,45,2026-03-24
1,AAPL,sale,Ensemble,2026-06-30,2028-03-31,8,2026-03-24
2,AAPL,sale,ETS,2026-06-30,2028-03-31,8,2026-03-24
3,AAPL,sale,LSTM,2026-06-30,2028-03-31,8,2026-03-24
4,AAPL,sale,Prophet,2026-06-30,2028-03-31,8,2026-03-24
5,AAPL,sale,SARIMA,2026-06-30,2028-03-31,8,2026-03-24
6,AAPL,sale,Theta,2026-06-30,2028-03-31,8,2026-03-24
7,MSFT,sale,actual,2015-03-31,2026-03-31,45,2026-03-24
8,MSFT,sale,Ensemble,2026-06-30,2028-03-31,8,2026-03-24
9,MSFT,sale,ETS,2026-06-30,2028-03-31,8,2026-03-24
